In [1]:
import os
os.chdir('../../..')

In [2]:
from src.datasets import QM9Dataset

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
import pandas as pd
import re

from rdkit import Chem
from rdkit.Chem import Draw
from collections import Counter
from torch_geometric.datasets import QM9
from scipy.stats import ks_2samp

In [5]:
qm9 = QM9Dataset(limit=None)
df = qm9.load()

2026-04-28 09:06:27.436 | INFO     | src.datasets:load:864 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-04-28 09:06:27.960 | INFO     | src.datasets:_sample_qm9_df:1064 - QM9 sampling complete: strategy=stratified, requested_limit=None, returned_rows=122608.
2026-04-28 09:06:27.965 | INFO     | src.datasets:_add_requested_descriptors:202 - Applying requested QM9 descriptors to sampled dataframe (rows=122608).
2026-04-28 09:06:27.966 | INFO     | src.datasets:_add_requested_descriptors:229 - No new descriptor columns added (already present or none requested).


In [12]:
qm9 = QM9(root='Anomaly-Detection-in-Molecular-and-Materials-Datasets/data/QM9/processed')

Extracting Anomaly-Detection-in-Molecular-and-Materials-Datasets/data/QM9/processed/raw/qm9.zip
Processing...
100%|██████████| 133885/133885 [00:38<00:00, 3473.09it/s]
Done!


In [19]:
data = qm9[0]
data.smiles

'[H]C([H])([H])[H]'

# Smiles and canonical smiles

In [8]:
df.filter(pl.col('smiles') != pl.col('canonical_smiles'))

mol_id,formula,smiles,canonical_smiles,scaffold_smiles,generic_scaffold,root_scaffold,brics_fragments,scaffold_tree_nodes,selfies,functional_groups,structure_class,is_injected,outlier_category,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,num_fluorine,num_heteroatoms,num_atoms,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,coordinates,atomic_numbers,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C
str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,i64,i64,f64,f64,i64,i64,i64,i64,i64,i64,f64,i64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,list[list[f64]],list[i64],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""qm9_22""","""C4H6""","""[H]CC#CC[H]""","""[H]C([H])([H])C#CC([H])([H])[H…","""Acyclic""","""Acyclic""","""Acyclic""","""[H]C([H])([H])C#CC([H])([H])[H…","""""","""[H][C][Branch1][C][H][Branch1]…","""""","""Acyclic""",0,null,54,1,0,0.957365,12.663176,4,0,0,0,0,10,1.8,0,0.5,0.0,0.5,0,0,2,2,0,2,5,18,1.203994,0,0,0,0,0,0,0,0,0,0,"[[0.681, 0.0, 0.0], [-0.681, 0.0, 0.0], … [2.9496, 0.0, 0.0]]","[6, 6, … 1]",0.0,38.52,-7.072239,-0.582324,6.492637,278.626404,1.016454,-4175.85498,-4175.735352,-4175.709473,-4176.474609,15.312,-29.176378,-29.287834,-29.416328,-27.633137,0.0,4.425972,4.425972
"""qm9_23""","""C3H6N+""","""[H]CC#C[NH3+]""","""[H]C([H])([H])C#C[N+]([H])([H]…","""Acyclic""","""Acyclic""","""Acyclic""","""[H]C([H])([H])C#C[N+]([H])([H]…","""""","""[H][C][Branch1][C][H][Branch1]…","""""","""Acyclic""",0,null,56,0,27,0.691153,12.99056,4,0,0,0,1,10,1.8,0,0.666667,0.0,0.333333,1,0,2,2,0,1,5,18,1.176148,0,0,0,0,0,0,0,0,0,0,"[[0.0151, 0.0, 1.0], [1.3823, 0.0, 1.0], … [2.543, 0.0, 1.0]]","[6, 6, … 7]",3.792,32.66,-8.440972,-1.477578,6.963394,260.189606,0.741755,-4613.901367,-4613.79248,-4613.767578,-4614.534668,12.93,-25.396606,-25.480663,-25.583441,-24.143276,0.0,4.579322,4.579322
"""qm9_44""","""C3H8N+""","""[H]C([H])([H])[NH+]1C([H])([H]…","""[H]C([H])([H])[N+]1([H])C([H])…","""C1C[NH2+]1""","""C1CC1""","""*1*[*+]1""","""[H]C([H])([H])[N+]1([H])C([H])…","""[H]C([H])([H])[N+]1([H])C([H])…","""[H][C][Branch1][C][H][Branch1]…","""""","""Aliphatic Ring""",0,null,58,-1,4,0.70166,13.091873,4,1,0,0,1,12,2.0,1,0.0,0.0,1.0,1,0,4,0,0,3,4,26,1.220415,0,0,0,0,0,0,0,0,0,0,"[[-0.0528, 1.4742, 0.01], [0.0194, 0.0219, 0.0633], … [0.3701, -1.4402, 1.6223]]","[6, 7, … 1]",1.1353,39.02,-6.269503,2.634062,8.900845,270.550812,2.657763,-4711.591309,-4711.470215,-4711.444336,-4712.306641,15.298,-41.40741,-41.710541,-41.967609,-38.496933,16.58914,7.18798,6.11415
"""qm9_49""","""C4H5N""","""[H]C1=C([H])N([H])C([H])=C1[H]""","""[H]c1c([H])c([H])n([H])c1[H]""","""c1cc[nH]c1""","""C1CCCC1""","""*1:*:*:*:*:1""","""[H]c1c([H])c([H])n([H])c1[H]""","""[H]c1c([H])c([H])n([H])c1[H],c…","""[H][C][C][Branch1][C][H][=C][B…","""""","""Aromatic""",0,null,67,1,15,0.7419451,12.756745,5,1,1,0,1,10,2.0,0,0.0,1.0,0.0,1,0,5,0,4,0,4,18,1.224717,0,0,0,0,0,0,0,0,0,0,"[[-0.0083, 1.3536, 0.01], [1.2803, 1.8246, -0.0002], … [-0.9255, -0.577, 0.0084]]","[7, 6, … 1]",1.8689,43.139999,-5.52119,1.357848,6.879038,303.980804,2.243116,-5717.160645,-5717.052734,-5717.027344,-5717.878906,14.821,-44.340279,-44.617588,-44.848911,-41.616554,9.17136,9.04195,4.5531
"""qm9_50""","""C3H4N2""","""[H]C1=NC([H])=C([H])N1[H]""","""[H]c1nc([H])n([H])c1[H]""","""c1c[nH]cn1""","""C1CCCC1""","""*1:*:*:*:*:1""","""[H]c1nc([H])n([H])c1[H]""","""[H]c1nc([H])n([H])c1[H],c1c[nH…","""[H][C][N][=C][Branch1][C][H][N…","""""","""Aromatic""",0,null,68,0,28,0.444793,13.026985,5,1,1,0,2,9,2.0,0,0.0,1.0,0.0,1,1,4,0,3,0,4,15,1.23137,0,0,0,0,0,0,0,0,0,0,"[[-0.01, 1.3564, 0